# In-database model pipeline and reranking

This notebook automates the complete in-database reranking workflow: download a supported model from Hugging Face, convert and validate it as an ONNX artifact, upload it to OCI Object Storage, load it into Oracle AI Database via the Oracle VecDB Python SDK, prove in-database reranking works, and revoke the temporary PAR.

Run sections 1 through 10 in order. Section 11 is optional cleanup. If you rerun the notebook with the same model name, run Section 11 first to remove the previous database model.

## Workflow at a glance

![In-database reranking workflow](images/workflow.svg)

The OCI CLI is used for Object Storage and PAR operations. The model conversion runs in the container, and model loading and reranking run in the database.

## Install notebook dependencies

Run this cell once in the notebook's Python environment. It installs the packages used by the host-side workflow. The conversion dependencies are installed inside the Docker or Podman container in Section 3.

In [ ]:
%pip install --upgrade oracle-vecdb python-dotenv

## Prerequisites

Prepare these items before running Sections 1 through 10:

- **Notebook host:** Python 3.10 or later and Jupyter. You can run the notebook from the repository root, this notebook's directory, or another directory inside the repository.
- **Host packages:** Run the **Install notebook dependencies** cell above once. It installs `oracle-vecdb` and `python-dotenv`.
- **Container runtime:** Install [Docker Desktop](https://docs.docker.com/desktop/) or [Podman](https://podman.io/docs/installation), then start Docker Desktop or a Podman machine. Allocate at least 12 GB to the container virtual machine for conversion.
- **OCI CLI:** Install the [OCI CLI](https://docs.oracle.com/en-us/iaas/Content/API/SDKDocs/cliinstall.htm) and configure a profile that can upload to Object Storage and manage PARs.
- **OML4Py archive:** Download Oracle's [`oml4py-client-linux-x86_64-2.1.1.zip`](https://www.oracle.com/database/technologies/oml4py-downloads.html) archive.
- **Database access:** Use Oracle AI Database 26ai or later with a compatible VecDB REST service, valid credentials, and permission to load and rerank models.
- **Configuration:** Copy `notebooks/vecdb/reranking/.env.example` to `notebooks/vecdb/reranking/.env` and set the VecDB, container, archive, OCI config, profile, and bucket values.
- **Model:** Use the tested `MODEL_ID=BAAI/bge-reranker-base`. Other Hugging Face rerankers require a separate custom conversion workflow.

The host-side install cell does not install Docker, Podman, or the OCI CLI. The conversion runs in a public Linux x86_64 Python 3.13 image, not the licensed OML4Py container; Apple Silicon and Windows use x86_64 emulation. The Object Storage namespace is discovered automatically. Leave `OCI_COMPARTMENT_NAME` blank for the tenancy/root compartment, or set an exact child-compartment name when a new bucket is created.

## 1. Select a supported model and read configuration

This notebook uses Oracle OML4Py's preconfigured reranking pipeline for `BAAI/bge-reranker-base`, Oracle's documented and tested example. Set `MODEL_NAME` in `.env`; leave `MODEL_ID=BAAI/bge-reranker-base`.

If you are selecting a different model for a custom conversion, use this checklist:

1. Search Hugging Face for `reranker` or `cross-encoder`.
2. Open the model card and confirm it is a pairwise text-classification model that returns one relevance score—not an embedding model, generative model, or listwise reranker.
3. In the model's **Files and versions** tab, open `config.json` and look for `model_type: "xlm-roberta"` and `num_labels: 1`. The tokenizer must be `XLMRobertaTokenizer`, and the repository should contain `safetensors` weights.
4. The exact `owner/repository` ID must be supported by Oracle's custom `ONNXPipelineConfig` workflow. The preconfigured-model list printed in Section 3 is informational and includes embedding models as well as rerankers; appearing in that list does not make a model a reranker for this notebook.

This notebook intentionally does not accept arbitrary model IDs: its conversion code is fixed to the tested preconfigured BGE reranker.

This cell loads the settings and checks the local prerequisites.

In [ ]:
import configparser
import json
import os
import subprocess
from datetime import datetime, timedelta, timezone
from pathlib import Path

from dotenv import load_dotenv

search_roots = (Path.cwd(), *Path.cwd().parents)
env_candidates = [
    root / "notebooks/vecdb/reranking/.env" for root in search_roots
] + [root / ".env" for root in search_roots]
ENV_FILE = next((path for path in env_candidates if path.is_file()), None)
if ENV_FILE is None:
    expected = Path("notebooks/vecdb/reranking/.env")
    raise FileNotFoundError(
        f"Could not find {expected}. Copy {expected.with_name('.env.example')} to {expected} first."
    )
repo = next(
    (root for root in search_roots
     if (root / "notebooks/vecdb/reranking/.env").resolve() == ENV_FILE.resolve()),
    ENV_FILE.parent,
)
load_dotenv(ENV_FILE, override=True)


def setting(name, default=""):
    return os.environ.get(name, default).strip()


CONTAINER_CLI = setting("CONTAINER_CLI", "podman").lower()
CONTAINER_PLATFORM = setting("RERANK_CONTAINER_PLATFORM", "linux/amd64")
CONVERSION_IMAGE = setting(
    "RERANK_CONVERSION_IMAGE", "mirror.gcr.io/library/python:3.13.5-slim-bookworm"
)
CONTAINER_NAME = "vecdb-reranker-conversion"
OML4PY_ARCHIVE = setting("OML4PY_ARCHIVE")
MODEL_ID = setting("MODEL_ID", "BAAI/bge-reranker-base")
MODEL_NAME = setting("MODEL_NAME", "BGE_RERANKER_BASE_INT8")
OCI_CONFIG_FILE = setting("OCI_CONFIG_FILE")
OCI_PROFILE = setting("OCI_PROFILE", "DEFAULT")
OCI_BUCKET = setting("OCI_BUCKET")
OCI_COMPARTMENT_NAME = setting("OCI_COMPARTMENT_NAME")
VECDB_REST_URL = setting("VECDB_REST_URL")
VECDB_ACCESS_TOKEN = setting("VECDB_ACCESS_TOKEN")
VECDB_USERNAME = setting("VECDB_USERNAME")
VECDB_PASSWORD = setting("VECDB_PASSWORD")
VECDB_SELF_SIGNED_SSL = setting("VECDB_SELF_SIGNED_SSL", "false").lower() == "true"
MAX_SEQ_LENGTH = int(setting("RERANK_MAX_SEQ_LENGTH", "256"))
MODEL_FILE_SETTING = setting("RERANK_MODEL_FILE")
OBJECT_NAME = setting("RERANK_OBJECT_NAME")
PAR_NAME = setting("RERANK_PAR_NAME")


def local_path(value, default):
    path = Path(value).expanduser() if value else default
    return (path if path.is_absolute() else repo / path).resolve()


MODEL_FILE = local_path(
    MODEL_FILE_SETTING,
    repo / "notebooks/vecdb/reranking/artifacts/reranking" / f"{MODEL_NAME.lower()}.onnx",
)
OUTPUT_DIR = MODEL_FILE.parent
OUTPUT_STEM = MODEL_FILE.stem
CONTAINER_ARCHIVE = "/input/oml4py-client-linux-x86_64-2.1.1.zip"

if CONTAINER_CLI not in {"podman", "docker"}:
    raise ValueError("CONTAINER_CLI must be podman or docker")
if not Path(OML4PY_ARCHIVE).expanduser().is_file():
    raise FileNotFoundError("Set OML4PY_ARCHIVE in .env to the downloaded Oracle archive")
if not Path(OCI_CONFIG_FILE).expanduser().is_file():
    raise FileNotFoundError("Set OCI_CONFIG_FILE in .env to your OCI config file")
if not VECDB_ACCESS_TOKEN and (not VECDB_USERNAME or not VECDB_PASSWORD):
    raise ValueError("Set VECDB_ACCESS_TOKEN or VECDB_USERNAME and VECDB_PASSWORD in .env")
if not VECDB_REST_URL or VECDB_REST_URL.startswith("<"):
    raise ValueError("Set VECDB_REST_URL in .env")
if not OCI_BUCKET or OCI_BUCKET.startswith("<"):
    raise ValueError("Set OCI_BUCKET in .env")

print(f"Model: {MODEL_ID} -> {MODEL_NAME}")
print(f"Container: {CONTAINER_CLI} ({CONTAINER_PLATFORM})")
print(f"Output: {MODEL_FILE}")

## 2. Prepare the conversion container

The container is an ordinary public Python image. It is not the licensed OML4Py image and does not require an Oracle Container Registry login. The setup is issued through Python subprocesses instead of `%%bash` so the same notebook works from macOS, Windows, Docker, and Podman.

Section 2 creates one persistent container named `vecdb-reranker-conversion` and reuses it in later sections. If you change the archive path, output path, image, or platform in `.env`, remove that container with the printed command and rerun this section so the mounted paths are refreshed.

In [ ]:
def run(command, *, capture=False, check=True):
    try:
        return subprocess.run(
            command, text=True, capture_output=capture, check=check
        )
    except FileNotFoundError as exc:
        raise RuntimeError(f"Required executable was not found: {command[0]}") from exc


def container(*args, capture=False, check=True):
    return run([CONTAINER_CLI, *args], capture=capture, check=check)


def bind_mount(path, target, readonly=False):
    suffix = ",readonly" if readonly else ""
    return f"type=bind,source={Path(path).resolve()},target={target}{suffix}"


if CONTAINER_CLI == "podman":
    container("machine", "start", capture=True, check=False)
info = container("info", capture=True, check=False)
if info.returncode:
    detail = (info.stderr or info.stdout or "").strip()
    raise RuntimeError(f"{CONTAINER_CLI} engine is not ready: {detail}")
print(f"{CONTAINER_CLI} engine is ready.")

image = container(
    "image", "inspect", "--format", "{{.Architecture}}", CONVERSION_IMAGE,
    capture=True, check=False,
)
if image.returncode or image.stdout.strip() not in {"amd64", "x86_64"}:
    print(f"Pulling {CONVERSION_IMAGE} for {CONTAINER_PLATFORM}")
    container("pull", "--platform", CONTAINER_PLATFORM, CONVERSION_IMAGE)

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
existing = container(
    "container", "inspect", CONTAINER_NAME, capture=True, check=False
)
if existing.returncode:
    container(
        "container", "create", "--name", CONTAINER_NAME,
        "--platform", CONTAINER_PLATFORM,
        "--env", "HF_HOME=/work/hf-cache",
        "--mount", bind_mount(OUTPUT_DIR, "/work"),
        "--mount", bind_mount(OML4PY_ARCHIVE, CONTAINER_ARCHIVE, readonly=True),
        CONVERSION_IMAGE, "sleep", "infinity",
    )
    container("container", "start", CONTAINER_NAME)
else:
    running = container(
        "container", "inspect", "--format", "{{.State.Running}}", CONTAINER_NAME,
        capture=True,
    ).stdout.strip()
    if running != "true":
        container("container", "start", CONTAINER_NAME)
    print(f"Reusing {CONTAINER_NAME}. If its archive, output path, image, or platform changed, remove it with: {CONTAINER_CLI} container rm --force {CONTAINER_NAME}")

print(f"Ready: {CONTAINER_NAME}")

## 3. Install the conversion dependencies

This installs the documented conversion packages and the official OML4Py slim client wheel inside the public container. The wheel is extracted from the downloaded archive without using the OML4Py container. The final part confirms that this notebook is using its tested BGE reranker in Oracle's current preconfigured catalog.

In [ ]:
requirements = [
    "numpy==2.1.0",
    "onnxruntime==1.20.0",
    "onnxruntime-extensions==0.14.0",
    "onnx==1.18.0",
    "torch==2.9.0",
    "transformers==4.56.1",
    "sentencepiece==0.2.1",
    "Pillow",
]

ready = container(
    "exec", CONTAINER_NAME, "python", "-c",
    "from oml.utils import MiningFunction, ONNXPipeline, ONNXPipelineConfig",
    capture=True, check=False,
)
if ready.returncode:
    container(
        "exec", CONTAINER_NAME, "python", "-m", "pip", "install",
        "--no-cache-dir", "--extra-index-url", "https://download.pytorch.org/whl/cpu",
        *requirements,
    )
    extract_wheel = """import pathlib, zipfile
archive = pathlib.Path(%r)
with zipfile.ZipFile(archive) as outer:
    wheel = next(name for name in outer.namelist()
                 if name.startswith('client/slim/') and name.endswith('.whl'))
    target = pathlib.Path('/tmp') / pathlib.Path(wheel).name
    target.write_bytes(outer.read(wheel))
print(target)
""" % CONTAINER_ARCHIVE
    container("exec", CONTAINER_NAME, "python", "-c", extract_wheel)
    container(
        "exec", CONTAINER_NAME, "python", "-m", "pip", "install",
        "/tmp/oml-2.1.1-cp313-cp313-linux_x86_64.whl",
    )
    print("Conversion dependencies installed.")
else:
    print("Conversion dependencies already installed.")

preconfigured_code = (
    "import json; from oml.utils import ONNXPipelineConfig; "
    "print(json.dumps(ONNXPipelineConfig.show_preconfigured()))"
)
preconfigured = container(
    "exec", CONTAINER_NAME, "python", "-c", preconfigured_code,
    capture=True,
)
preconfigured_models = json.loads(preconfigured.stdout.strip())
if MODEL_ID != "BAAI/bge-reranker-base":
    raise ValueError(
        f"{MODEL_ID!r} is not supported by this notebook. Set MODEL_ID=BAAI/bge-reranker-base, "
        "or adapt Oracle's custom ONNXPipelineConfig workflow for another model."
    )
if MODEL_ID not in preconfigured_models:
    raise RuntimeError("Oracle's installed OML4Py client no longer lists the tested BGE reranker.")
print(f"Using Oracle preconfigured reranker: {MODEL_ID}")


## 4. Convert the model to ONNX

This is the documented OML4Py conversion API. It downloads the model from Hugging Face inside the container, applies the preconfigured reranking pipeline, quantizes it, and writes the ONNX file into the host output directory.

In [ ]:
conversion_code = f"""
import time
from huggingface_hub import snapshot_download
from oml.utils import MiningFunction, ONNXPipeline, ONNXPipelineConfig

model_id = {MODEL_ID!r}
started = time.monotonic()
print(f"Downloading {{model_id}} from Hugging Face...", flush=True)
snapshot_download(model_id)
print("Download complete; preparing the Oracle pipeline...", flush=True)
config = ONNXPipelineConfig.from_preconfigured(
    model_id,
    max_seq_length={MAX_SEQ_LENGTH},
    quantize_model=True,
    use_float16=False,
)
pipeline = ONNXPipeline(
    model_id,
    config=config,
    function=MiningFunction.REGRESSION,
)
print("Exporting and quantizing the ONNX pipeline...", flush=True)
pipeline.export2file({OUTPUT_STEM!r}, "/work")
print(f"Export complete in {{time.monotonic() - started:.1f}} seconds.", flush=True)
print(pipeline.summary())
"""
container("exec", CONTAINER_NAME, "python", "-c", conversion_code)
if not MODEL_FILE.is_file() or MODEL_FILE.stat().st_size == 0:
    raise RuntimeError(f"Conversion did not create {MODEL_FILE}")
print(f"Created {MODEL_FILE} ({MODEL_FILE.stat().st_size:,} bytes)")


## 5. Validate the ONNX file

Run one independent ONNX Runtime inference before uploading. The exported graph includes Oracle's paired tokenizer, so the validation registers the `onnxruntime-extensions` custom-operator library.

In [ ]:
validation_code = f"""
import onnx
import onnxruntime as ort
from onnxruntime_extensions import get_library_path

model_path = {f"/work/{MODEL_FILE.name}"!r}
model = onnx.load(model_path)
onnx.checker.check_model(model)
options = ort.SessionOptions()
options.register_custom_ops_library(get_library_path())
session = ort.InferenceSession(
    model_path, options, providers=["CPUExecutionProvider"]
)
result = session.run(
    None,
    {{
        "first_input": ["What is Oracle Vector Database?"],
        "second_input": ["Oracle Vector Database supports vector search and reranking."],
    }},
)[0]
if tuple(result.shape) != (1, 1):
    raise RuntimeError(f"Expected one score, got shape {{result.shape}}")
print("ONNX validation output:", result.tolist())
"""
container("exec", CONTAINER_NAME, "python", "-c", validation_code)
print("ONNX file passed structural and one-pair inference validation.")

## 6. Upload the model to OCI Object Storage

The bucket is created automatically when it does not exist. Leave `OCI_COMPARTMENT_NAME` blank to create it in the tenancy/root compartment, or set the exact child-compartment name in `.env`.

In [ ]:
def oci(*args, capture=True, check=True):
    return run(
        ["oci", *args, "--config-file", str(Path(OCI_CONFIG_FILE).expanduser()),
         "--profile", OCI_PROFILE],
        capture=capture, check=check,
    )


def config_value(name):
    parser = configparser.ConfigParser()
    parser.read(Path(OCI_CONFIG_FILE).expanduser())
    section = OCI_PROFILE if parser.has_section(OCI_PROFILE) else "DEFAULT"
    value = parser.get(section, name, fallback="").strip()
    if not value:
        raise ValueError(f"OCI profile {OCI_PROFILE!r} has no {name} value")
    return value


def compartment_id():
    tenancy = config_value("tenancy")
    if not OCI_COMPARTMENT_NAME:
        return tenancy
    items = json.loads(oci(
        "iam", "compartment", "list", "--compartment-id", tenancy,
        "--compartment-id-in-subtree", "true", "--access-level", "ACCESSIBLE", "--all",
    ).stdout).get("data", [])
    matches = [item["id"] for item in items
               if item.get("name") == OCI_COMPARTMENT_NAME
               and str(item.get("lifecycle-state", "")).upper() != "DELETED"
               and item.get("id")]
    if not matches:
        raise ValueError(f"OCI compartment not found or not accessible: {OCI_COMPARTMENT_NAME}")
    if len(matches) > 1:
        raise ValueError(f"OCI compartment name is not unique: {OCI_COMPARTMENT_NAME}")
    return matches[0]


def ensure_bucket(namespace):
    check = oci(
        "os", "bucket", "get", "--namespace-name", namespace,
        "--bucket-name", OCI_BUCKET, check=False,
    )
    if check.returncode == 0:
        return
    detail = (check.stderr or check.stdout or "").lower()
    if "404" not in detail and "notfound" not in detail and "not found" not in detail:
        raise RuntimeError(f"Could not check bucket {OCI_BUCKET}: {detail}")
    oci(
        "os", "bucket", "create", "--namespace-name", namespace,
        "--compartment-id", compartment_id(), "--name", OCI_BUCKET,
    )
    print(f"Created Object Storage bucket: {OCI_BUCKET}")


namespace = oci("os", "ns", "get", "--query", "data", "--raw-output").stdout.strip()
if not namespace:
    raise RuntimeError("OCI CLI returned an empty Object Storage namespace")
if not OBJECT_NAME:
    OBJECT_NAME = f"vecdb-rerank/{MODEL_NAME.lower()}.onnx"
ensure_bucket(namespace)
oci(
    "os", "object", "put", "--namespace-name", namespace,
    "--bucket-name", OCI_BUCKET, "--name", OBJECT_NAME,
    "--file", str(MODEL_FILE), "--force",
)
remote_size = oci(
    "os", "object", "head", "--namespace-name", namespace,
    "--bucket-name", OCI_BUCKET, "--name", OBJECT_NAME,
    "--query", '"content-length"', "--raw-output",
).stdout.strip()
if remote_size != str(MODEL_FILE.stat().st_size):
    raise RuntimeError(
        f"Object size mismatch: local={MODEL_FILE.stat().st_size}, remote={remote_size}"
    )
print(f"Uploaded {OBJECT_NAME} ({remote_size} bytes)")

## 7. Create a short-lived PAR

The database reads the private model object through this object-scoped, four-hour PAR. The cell prints the URL so you can inspect or copy it. Treat the URL as a secret until it expires or is revoked.

In [ ]:
if not PAR_NAME:
    PAR_NAME = f"{MODEL_NAME.lower()}-par-{datetime.now(timezone.utc):%Y%m%d%H%M%S}"
expiry = (datetime.now(timezone.utc) + timedelta(hours=4)).strftime("%Y-%m-%dT%H:%M:%SZ")
par_data = json.loads(oci(
    "os", "preauth-request", "create", "--namespace-name", namespace,
    "--bucket-name", OCI_BUCKET, "--name", PAR_NAME,
    "--object-name", OBJECT_NAME, "--access-type", "ObjectRead",
    "--time-expires", expiry,
).stdout)["data"]
par_id = par_data["id"]
par_url = f"https://objectstorage.{config_value('region')}.oraclecloud.com{par_data['access-uri']}"
print(f"Created object-scoped PAR {PAR_NAME}; it expires in four hours.")
print(f"PAR URL: {par_url}")

## 8. Load the model into VecDB

The model is loaded as a regression pipeline. The metadata names the generated ONNX output `output`. If this section fails, run Section 10 to revoke the PAR before retrying. If the error says the model name already exists, run Section 11 first, then rerun from Section 7 with a fresh PAR.

In [ ]:
def connect_vecdb():
    from oracle_vecdb import Configuration, OracleVecDB
    kwargs = {"rest_url": VECDB_REST_URL}
    if VECDB_ACCESS_TOKEN:
        kwargs["access_token"] = VECDB_ACCESS_TOKEN
    else:
        kwargs.update(username=VECDB_USERNAME, password=VECDB_PASSWORD)
    config = Configuration(**kwargs)
    if VECDB_SELF_SIGNED_SSL:
        config.verify_ssl = False
    return OracleVecDB(config)


vecdb = connect_vecdb()
print(vecdb.load_model(
    model_name=MODEL_NAME,
    url=par_url,
    model_params={
        "metadata": {"function": "regression", "regressionOutput": "output"}
    },
))

## 9. Verify the model and prove reranking works

The final request sends one query and three candidate documents. The Oracle-related document must score above the unrelated cake recipe.

In [ ]:
models = json.loads(vecdb.list_models().to_json())
print("Loaded models:", [
    item.get("model_name", item.get("modelName"))
    for item in models.get("items", [])
])
print("Model description:")
print(vecdb.describe_model(model_name=MODEL_NAME))

documents = [
    "A reranking model scores candidate documents for relevance to a query.",
    "A recipe for chocolate cake uses flour, eggs, and cocoa.",
    "Oracle Vector Database supports vector search and reranking.",
]
result = vecdb.rerank(
    query="How does Oracle Vector Database rerank search results?",
    documents=documents,
    model_name=MODEL_NAME,
)
scores = {}
for rank, item in enumerate(result.items or [], 1):
    index, score = int(item.index), float(item.score)
    scores[index] = score
    print(f"{rank}. score={score:.6f}  {documents[index]}")
if scores.get(2, float("-inf")) <= scores.get(1, float("-inf")):
    raise RuntimeError("The Oracle-related document did not outrank the recipe")
print("Success: the loaded model returned live reranking scores.")

## 10. Revoke the PAR

After the load attempt, run this section—even when loading failed—to revoke the temporary URL. The model object remains in OCI Object Storage. If you retry, rerun Section 7 first to create a fresh PAR.

In [ ]:
if globals().get("par_id"):
    revoked = oci(
        "os", "preauth-request", "delete", "--namespace-name", namespace,
        "--bucket-name", OCI_BUCKET, "--par-id", par_id, "--force",
        check=False,
    )
    if revoked.returncode:
        detail = (revoked.stderr or revoked.stdout or "").lower()
        if "not found" not in detail and "404" not in detail:
            raise RuntimeError(f"Could not revoke PAR: {detail}")
        print("PAR was already revoked or expired.")
    else:
        print("Revoked temporary PAR; the model object remains in Object Storage.")
else:
    print("No PAR was created in this notebook session.")

## 11. Optional cleanup: remove the model from VecDB

This standalone cell reloads `.env`, prints only the model names, drops `MODEL_NAME` when it is present, and prints the remaining names. It does not delete the model object from OCI Object Storage.

In [ ]:
import json
import os
from pathlib import Path
from dotenv import load_dotenv
from oracle_vecdb import Configuration, OracleVecDB

search_roots = (Path.cwd(), *Path.cwd().parents)
env_candidates = [
    root / "notebooks/vecdb/reranking/.env" for root in search_roots
] + [root / ".env" for root in search_roots]
env_file = next((path for path in env_candidates if path.is_file()), None)
if env_file is None:
    expected = Path("notebooks/vecdb/reranking/.env")
    raise FileNotFoundError(
        f"Could not find {expected}. Copy {expected.with_name('.env.example')} to {expected} first."
    )
load_dotenv(env_file, override=True)
config_args = {"rest_url": os.environ["VECDB_REST_URL"]}
if os.environ.get("VECDB_ACCESS_TOKEN"):
    config_args["access_token"] = os.environ["VECDB_ACCESS_TOKEN"]
else:
    config_args["username"] = os.environ["VECDB_USERNAME"]
    config_args["password"] = os.environ["VECDB_PASSWORD"]
config = Configuration(**config_args)
if os.environ.get("VECDB_SELF_SIGNED_SSL", "").lower() == "true":
    config.verify_ssl = False
db = OracleVecDB(config)


def model_names():
    data = json.loads(db.list_models().to_json())
    return [item.get("model_name", item.get("modelName"))
            for item in data.get("items", [])]


before = model_names()
print(before)
model_name = os.environ.get("MODEL_NAME", "BGE_RERANKER_BASE_INT8")
if model_name in before:
    db.drop_model(model_name=model_name)
print(model_names())